In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, avg, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas7-BigData") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/10/11 21:49:42 WARN Utils: Your hostname, ubuntu resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/10/11 21:49:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/11 21:49:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/11 21:49:44 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 3.5.9


In [4]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


### Instruksi Pengerjaan

Buat notebook baru **`Tugas7_[NPM]_[Nama Lengkap].ipynb`**, lalu bangun pipeline analisis lengkap:

---

**A. EXTRACT** *(bobot 10%)*

Baca ketiga sumber data **dari HDFS** (bukan disk lokal) menjadi tiga Spark DataFrame.

**B. TRANSFORM — Join & Kolom Turunan** *(bobot 20%)*

Gabungkan ketiga tabel, tambahkan kolom `total_penjualan` (`qty x harga`).

**C. TRANSFORM — Window Function** *(bobot 20%)*

Tentukan **top-2 menu terlaris** (berdasarkan `total_penjualan`) di **setiap cabang**, gunakan `row_number()`.

**D. TRANSFORM — Spark SQL** *(bobot 20%)*

Menggunakan **Spark SQL murni**, tampilkan total penjualan per `kepala_barista`, diurutkan dari tertinggi.

**E. LOAD** *(bobot 15%)*

Simpan data gabungan (hasil bagian B) ke HDFS sebagai Parquet, path `/user/[username]/tugas7/processed/laporan`, dipartisi berdasarkan `kategori_menu`. Verifikasi dengan membaca kembali dan menghitung jumlah baris.

**F. Ringkasan Eksekutif & Rekomendasi** *(bobot 15%)*

Buat tabel ringkasan (per cabang: total penjualan, jumlah transaksi, rata-rata nilai transaksi), lalu tulis pada markdown cell (**minimal 120 kata**): cabang mana yang berkinerja terbaik, menu apa yang sebaiknya dipromosikan lebih gencar di cabang dengan kinerja terlemah (berdasarkan temuan bagian C), dan alasannya.

---



In [5]:
!hdfs dfs -ls /user/mahasiswa/tugas7/raw

Found 3 items
-rw-r--r--   1 vboxuser supergroup        321 2026-10-11 21:55 /user/mahasiswa/tugas7/raw/tugas7_cabang.json
-rw-r--r--   1 vboxuser supergroup        432 2026-10-11 21:55 /user/mahasiswa/tugas7/raw/tugas7_menu.csv
-rw-r--r--   1 vboxuser supergroup      52539 2026-10-11 21:55 /user/mahasiswa/tugas7/raw/tugas7_transaksi.csv


In [7]:
# A. EXTRACT

base_path = "hdfs://localhost:9000/user/mahasiswa/tugas7/raw"

# Membaca data cabang dari HDFS
df_cabang = spark.read.json(
    f"{base_path}/tugas7_cabang.json"
)

# Membaca data menu dari HDFS
df_menu = spark.read.option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{base_path}/tugas7_menu.csv")

# Membaca data transaksi dari HDFS
df_transaksi = spark.read.option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{base_path}/tugas7_transaksi.csv")

print("Jumlah data cabang:", df_cabang.count())
print("Jumlah data menu:", df_menu.count())
print("Jumlah data transaksi:", df_transaksi.count())


Jumlah data cabang: 3
Jumlah data menu: 15
Jumlah data transaksi: 4000


In [8]:
# B. TRANSFROM


from pyspark.sql.functions import col

# Menggabungkan transaksi dengan data menu
df_transform = df_transaksi.join(
    df_menu,
    on="menu_id",
    how="inner"
)

# Menggabungkan hasilnya dengan data cabang
df_transform = df_transform.join(
    df_cabang,
    on="cabang_id",
    how="inner"
)

# Menghitung total penjualan setiap transaksi
df_transform = df_transform.withColumn(
    "total_penjualan",
    col("qty") * col("harga")
)

# Menampilkan hasil transformasi
df_transform.show(10, truncate=False)

print("Jumlah data hasil transformasi:", df_transform.count())
print("Daftar kolom:", df_transform.columns)


+---------+-------+------+---+---------+--------------+-----+--------------+----------+-------------------------+---------------+
|cabang_id|menu_id|trx_id|qty|nama_menu|kategori_menu |harga|kepala_barista|kota      |nama_cabang              |total_penjualan|
+---------+-------+------+---+---------+--------------+-----+--------------+----------+-------------------------+---------------+
|1        |14     |KN0   |4  |Menu-14  |Kopi          |18000|Nadia         |Magelang  |Kopi Nusantara Magelang  |72000          |
|1        |11     |KN1   |4  |Menu-11  |Kopi          |22000|Nadia         |Magelang  |Kopi Nusantara Magelang  |88000          |
|1        |11     |KN2   |3  |Menu-11  |Kopi          |22000|Nadia         |Magelang  |Kopi Nusantara Magelang  |66000          |
|3        |11     |KN3   |1  |Menu-11  |Kopi          |22000|Nadia         |Semarang  |Kopi Nusantara Semarang  |22000          |
|2        |12     |KN4   |4  |Menu-12  |Makanan Ringan|35000|Reza          |Yogyakarta|Kop

In [9]:
# C. WINDOW FUNCTION MENCARI 2 BARIS MENU TERLARIS SETIAP CABANG


from pyspark.sql.functions import sum as spark_sum, row_number
from pyspark.sql.window import Window

# Menghitung total kuantitas terjual per menu di setiap cabang
df_penjualan_menu = df_transform.groupBy(
    "cabang_id",
    "nama_cabang",
    "menu_id",
    "nama_menu",
    "kategori_menu"
).agg(
    spark_sum("qty").alias("total_qty_terjual"),
    spark_sum("total_penjualan").alias("total_omzet")
)

# Membuat ranking menu di setiap cabang
window_menu = Window.partitionBy("cabang_id").orderBy(
    col("total_qty_terjual").desc()
)

df_ranking_menu = df_penjualan_menu.withColumn(
    "peringkat",
    row_number().over(window_menu)
)

# Mengambil dua menu dengan penjualan kuantitas tertinggi per cabang
df_top2_menu = df_ranking_menu.filter(
    col("peringkat") <= 2
)

df_top2_menu.orderBy(
    "cabang_id", "peringkat"
).show(truncate=False)


+---------+-------------------------+-------+---------+--------------+-----------------+-----------+---------+
|cabang_id|nama_cabang              |menu_id|nama_menu|kategori_menu |total_qty_terjual|total_omzet|peringkat|
+---------+-------------------------+-------+---------+--------------+-----------------+-----------+---------+
|1        |Kopi Nusantara Magelang  |3      |Menu-3   |Makanan Ringan|260              |7800000    |1        |
|1        |Kopi Nusantara Magelang  |15     |Menu-15  |Non-Kopi      |246              |7380000    |2        |
|2        |Kopi Nusantara Yogyakarta|15     |Menu-15  |Non-Kopi      |272              |8160000    |1        |
|2        |Kopi Nusantara Yogyakarta|12     |Menu-12  |Makanan Ringan|265              |9275000    |2        |
|3        |Kopi Nusantara Semarang  |4      |Menu-4   |Non-Kopi      |268              |5896000    |1        |
|3        |Kopi Nusantara Semarang  |10     |Menu-10  |Kopi          |267              |4806000    |2        |
+

In [10]:
# D. SPARK SQL MENGHITUNG TOTAL PENJUALAN  KEPALA BARISTA

# Mendaftarkan hasil transformasi sebagai tabel sementara
df_transform.createOrReplaceTempView("penjualan_kopi")

# Menghitung omzet dan jumlah transaksi per kepala barista
hasil_sql = spark.sql("""
    SELECT
        kepala_barista,
        COUNT(DISTINCT trx_id) AS jumlah_transaksi,
        SUM(qty) AS total_item_terjual,
        SUM(total_penjualan) AS total_omzet
    FROM penjualan_kopi
    GROUP BY kepala_barista
    ORDER BY total_omzet DESC
""")

hasil_sql.show(truncate=False)


+--------------+----------------+------------------+-----------+
|kepala_barista|jumlah_transaksi|total_item_terjual|total_omzet|
+--------------+----------------+------------------+-----------+
|Nadia         |2695            |6734              |177680000  |
|Reza          |1305            |3205              |84645000   |
+--------------+----------------+------------------+-----------+



In [11]:
# E. LOAD MENYIMPAN DATA KE PARWUET DENGAN PARTISI KATEGORI MENU


# Lokasi penyimpanan hasil akhir di HDFS
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/penjualan_parquet"

# Menyimpan hasil transformasi ke format Parquet
df_transform.write \
    .mode("overwrite") \
    .partitionBy("kategori_menu") \
    .parquet(output_path)

print("Data berhasil disimpan dalam format Parquet.")


[Stage 40:>                                                         (0 + 1) / 1]

Data berhasil disimpan dalam format Parquet.


In [12]:
# VERIVIKASI HASIL PENYIMPNANA


# Membaca kembali data Parquet dari HDFS
df_parquet = spark.read.parquet(output_path)

print("Jumlah data yang dibaca kembali:", df_parquet.count())

# Memeriksa hasil pembacaan
df_parquet.select(
    "trx_id",
    "nama_cabang",
    "nama_menu",
    "kategori_menu",
    "total_penjualan"
).show(10, truncate=False)

# Memeriksa direktori partisi di HDFS
!hdfs dfs -ls /user/mahasiswa/tugas7/processed/penjualan_parquet


Jumlah data yang dibaca kembali: 4000
+------+-------------------------+---------+--------------+---------------+
|trx_id|nama_cabang              |nama_menu|kategori_menu |total_penjualan|
+------+-------------------------+---------+--------------+---------------+
|KN4   |Kopi Nusantara Yogyakarta|Menu-12  |Makanan Ringan|140000         |
|KN8   |Kopi Nusantara Semarang  |Menu-9   |Makanan Ringan|140000         |
|KN10  |Kopi Nusantara Magelang  |Menu-3   |Makanan Ringan|30000          |
|KN13  |Kopi Nusantara Yogyakarta|Menu-13  |Makanan Ringan|105000         |
|KN15  |Kopi Nusantara Semarang  |Menu-12  |Makanan Ringan|35000          |
|KN19  |Kopi Nusantara Yogyakarta|Menu-9   |Makanan Ringan|140000         |
|KN25  |Kopi Nusantara Magelang  |Menu-6   |Makanan Ringan|44000          |
|KN26  |Kopi Nusantara Semarang  |Menu-6   |Makanan Ringan|22000          |
|KN28  |Kopi Nusantara Semarang  |Menu-1   |Makanan Ringan|105000         |
|KN29  |Kopi Nusantara Magelang  |Menu-1   |Makana

In [14]:
# RINGKASAN EKSEKUTIF



from pyspark.sql.functions import sum as spark_sum, countDistinct, avg, round, col

# Ringkasan performa setiap cabang
df_ringkasan_cabang = df_transform.groupBy(
    "cabang_id", "nama_cabang"
).agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    countDistinct("trx_id").alias("jumlah_transaksi"),
    round(avg("total_penjualan"), 2).alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

df_ringkasan_cabang.show(truncate=False)


+---------+-------------------------+---------------+----------------+-------------------------+
|cabang_id|nama_cabang              |total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+---------+-------------------------+---------------+----------------+-------------------------+
|3        |Kopi Nusantara Semarang  |90542000       |1393            |64997.85                 |
|1        |Kopi Nusantara Magelang  |87138000       |1302            |66926.27                 |
|2        |Kopi Nusantara Yogyakarta|84645000       |1305            |64862.07                 |
+---------+-------------------------+---------------+----------------+-------------------------+



#  ANALISIS DAN REKOMENDASI 

### F. Ringkasan Eksekutif & Rekomendasi

Berdasarkan hasil analisis data penjualan Kopi Nusantara, cabang Semarang memiliki kinerja terbaik berdasarkan total penjualan, yaitu sebesar Rp90.542.000 dari 1.393 transaksi. Cabang Magelang menempati posisi kedua dengan total penjualan Rp87.138.000 dari 1.302 transaksi, sedangkan cabang Yogyakarta memiliki total penjualan terendah sebesar Rp84.645.000 dari 1.305 transaksi. Dengan demikian, cabang Yogyakarta menjadi prioritas evaluasi untuk meningkatkan kinerja penjualan. Meskipun demikian, rata-rata nilai transaksi tertinggi justru dimiliki cabang Magelang sebesar Rp66.926,27, diikuti Semarang sebesar Rp64.997,85 dan Yogyakarta sebesar Rp64.862,07.

Berdasarkan hasil analisis *Window Function* pada bagian C, dua menu dengan kuantitas penjualan tertinggi di cabang Yogyakarta adalah Menu-15 dengan total 272 item dan Menu-12 dengan total 265 item. Oleh karena itu, kedua menu tersebut direkomendasikan untuk dipromosikan lebih gencar melalui paket bundling, promosi pada jam ramai, atau penawaran khusus kepada pelanggan. Menu-12 juga memiliki omzet Rp9.275.000 di cabang Yogyakarta, sedangkan Menu-15 menghasilkan omzet Rp8.160.000. Promosi ini diharapkan dapat meningkatkan jumlah pembelian sekaligus mendorong pelanggan mencoba menu lain. Evaluasi penjualan secara berkala perlu dilakukan untuk mengetahui efektivitas promosi dan menentukan strategi lanjutan berdasarkan perubahan permintaan pelanggan.

In [ ]:
s